# INTRODUCTION

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
<h2>Global Tech Workforce Intelligence & Compensation Analysis</h2>
The modern technology job market is undergoing a rapid transformation driven by the expansion of Artificial Intelligence (AI), Machine Learning (ML), and remote work models. Understanding global compensation benchmarks, regional hiring demand, and critical skill combinations is essential for both employers and tech professionals.  This project utilizes the Open Global Workforce Intelligence Platform (OGWIP) dataset—a comprehensive repository combining real-world job posting data from YCombinator's HackerNews, federal public sector feeds (USAJOBS), and LLM-extracted metadata. This project establishes an empirical baseline of global hiring trends through exploratory data analysis and prepares structured feature representations to feed downstream predictive models.

# OJBECTIEVES:

<div style = "background-color:lightblue; color:green; font-size:20px; text-align:justify">
<ul>
<li>Primary Objectives (EDA Phase)
<ul>
<li>Compensation & Geographical Mapping: Uncover compensation distributions across key geographic hubs (North America, Europe, APAC) and analyze pay parity across remote, hybrid, and on-site roles.</li>
<li>Tech Stack & Skill Co-occurrence: Quantify market demand for key technical skills (Python, SQL, PyTorch, LLMs, Cloud platforms) and identify high-value skill clusters. </li>
<li> Seniority & Role Benchmark: Evaluate how salary scales across seniority levels (Junior to Staff/Principal) and role classifications (e.g., Data Scientist, DevOps Architect, Product Manager). </li>
</ul>
<li> Future Objectives (ML Phase)Salary Prediction Model (Regression):
<ul> 
<li>Predict continuous compensation figures (salary_in_usd) based on regional features, role category, experience level, and tech stack.</li>
<li>Skill Demand & Seniority Classification (Classification): Build multiclass classifiers to predict job seniority or role tiers using tech stack features and text representation.
<li>Skill Association Mining (NLP / Clustering): Extract latent skill patterns and grouping using topic modeling or unsupervised clustering to track emerging AI tech stacks.
</ul>
</ul>


</div>

<div>

# 1. Getting Ready with Dataset

In [2]:
import numpy as np
import pandas as pd
import polars as pl

import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
data = pl.read_csv('/home/kartika/Datasets/global_tech_market_2026.csv')
data = data.to_pandas()
data.head()

,job_id,job_title,company_name,location,salary_min_usd,salary_max_usd,tech_stack,source,description,date_posted
0,OGWIP_100001,Product Manager,European Space Agency,"Austin, TX, USA",81405,113143,"Python, SQL, TensorFlow, PyTorch",HackerNews,"We are European Space Agency, looking for a Pr...",2026-08-04 05:00:00
1,OGWIP_100003,NLP Engineer,Federal Bureau of Investigation,Remote - US Only,74906,118550,"Java, Spring Boot, Kafka, PostgreSQL",LinkedIn_Proxy,"We are Federal Bureau of Investigation, lookin...",2026-08-01 13:00:00
2,OGWIP_100004,Lead Data Scientist,AI Solutions LLC,Remote - Global,144270,215864,"Go, Kubernetes, Docker, GCP",LinkedIn_Proxy,"We are AI Solutions LLC, looking for a Lead Da...",2026-08-04 21:00:00
3,OGWIP_100005,Full Stack Developer,NASA,"Seattle, WA, USA",109716,165103,"Python, Airflow, Snowflake, dbt",HackerNews,"We are NASA, looking for a Full Stack Develope...",2026-08-02 22:00:00
4,OGWIP_100006,Technical Program Manager,Quantum Data,"Amsterdam, Netherlands",73276,111586,"JavaScript, React, Node.js, MongoDB",HackerNews,"We are Quantum Data, looking for a Technical P...",2026-08-02 05:00:00




## 3. ML-Oriented EDA Method

To ensure the exploratory phase directly lays the groundwork for future Machine Learning models, the analysis is structured into four sequential steps:

### Step 1: Data Health & Integrity Checks
1. **Target Variable Audit:** Examine the target variable (`salary_in_usd`) for skewness, zeroes, outliers, or missing values. Test log-transformations:
   $$\text{salary}_{\text{log}} = \log(1 + \text{salary}_{\text{usd}})$$
   to evaluate whether skewed salary distributions approach normality suitable for linear models.
2. **Missing Value & Imputation Strategy:** Assess missingness across numerical (compensation) and categorical (location, tech stack, experience) fields to decide on drop vs. imputation rules (e.g., KNN imputation, median by role/region).
3. **Data Type Parsing:** Parse raw JSON metadata, string lists (e.g., `['Python', 'AWS']`), and temporal features (`posting_date`) into explicit tabular arrays.

### Step 2: Feature & Target Relationship Analysis
1. **Univariate Analysis:** Plot distributions for numerical variables (histograms, boxplots) and frequency counts for categorical variables.
2. **Bivariate & Correlation Mapping:** Compute correlation matrices (Pearson/Spearman) between numeric attributes. Use box plots and ANOVA tests to check variance in salary across experience levels, company sizes, and work setups.
3. **High-Cardinality Exploration:** Analyze high-cardinality columns (e.g., specific job titles or locations) to design grouping strategies (e.g., mapping rare cities into regional hubs or top 20 categories + `"Other"`).

### Step 3: Text & Multi-Label Skill Processing
1. **Tech Stack One-Hot Encoding / TF-IDF:** Explode list-based tech stack columns into binary indicator variables (e.g., `has_pytorch`, `has_aws`).
2. **Co-occurrence Analysis:** Create heatmap matrices showing which skills appear together most frequently (e.g., `Python` + `PyTorch` vs. `React` + `TypeScript`).

### Step 4: Machine Learning Feature Engineering Readiness
1. **Encoding Selection:** Test One-Hot Encoding for low-cardinality categorical variables (`work_setup`, `experience_level`) versus Target/Frequency Encoding for high-cardinality columns (`country`, `exact_role`).
2. **Scaling Requirements:** Check feature ranges to determine normalization/standardization needs for distance-based models (e.g., Ridge, SVM, Neural Networks) vs. tree-based models (e.g., XGBoost, LightGBM).
3. **Train/Test Split Strategy:** Check if temporal trends exist (e.g., year-over-year salary shifts) to determine whether a standard random train/test split or a time-based split is required.